In [6]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# ============================
# CARREGAR TODOS OS CSVs
# ============================

results_path = Path("")

csv_files = list(results_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum CSV encontrado na pasta ../results")

dfs = []
for csv in csv_files:
    df_tmp = pd.read_csv(csv)
    df_tmp["source_file"] = csv.name  # opcional: rastrear origem
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# 🔧 garantir que erro é numérico
df["erro"] = pd.to_numeric(df["erro"], errors="coerce")

# ============================
# ORDENAR MODELOS
# ============================

order = (
    df.groupby("modelo")["erro"]
    .median()
    .sort_values()
    .index
    .tolist()
)

# ============================
# MAPA DE CORES
# ============================

color_map = {
    "baseline_ultra":  "#EF553B",
    "shape_ultra":  "#EF553B",
    "tail_ultra":  "#EF553B",
    "divergence_ultra": "#EF553B",
    "qderiv_ultra": "#EF553B",
    "baseline_tails_01": "#EF553B",
    "DyS_hellinger": "#fc03d3",
    "DyS_topsoe": "#fc03d3",
    "QuaDapt_DyS": "#fc03d3",
    "baseline_lite": "#19D3F3",
    "mfe": "#19D3F3",
    "qderiv_lite": "#19D3F3",
    "MiniRocket": "#19D3F3",
    "tsfresh": "#19D3F3",
    "catch22": "#19D3F3",
    "divergence_lite": "#19D3F3",
    "tail_lite": "#19D3F3",
    "shape_lite": "#19D3F3"
}

# ============================
# PLOT
# ============================

fig = px.box(
    df,
    x="modelo",
    y="erro",
    category_orders={"modelo": order},
    points="all",
    color="modelo",
    color_discrete_map=color_map
)

fig.update_traces(
    jitter=0.35,
    marker=dict(size=4, opacity=0.6),
)

fig.update_layout(
    title="Comparação de erro entre modelos (ordenado do melhor ao pior)",
    xaxis_title="Modelo",
    yaxis_title="Erro absoluto |prev_pred − prev_real|",
    template="simple_white",
    width=950,
    height=450,
    showlegend=True
)

fig

In [7]:
#!/usr/bin/env python3

import os
import pandas as pd


def load_csvs(csv_files):

    dfs = []

    for path in csv_files:

        if not os.path.exists(path):
            print(f"⚠️ Arquivo não encontrado: {path}")
            continue

        print(f"📂 Lendo: {path}")

        df = pd.read_csv(path)

        # opcional: adicionar origem
        df["source_file"] = os.path.basename(path)

        dfs.append(df)

    if not dfs:
        raise ValueError("Nenhum CSV válido encontrado.")

    return pd.concat(dfs, ignore_index=True)


def main():

    base_dir = os.getcwd()

    csv_files = [

        os.path.join(
            base_dir,
            "moss_calibrated_results.csv"
        ),

        os.path.join(
            base_dir,
            "quadapt.csv"
        ),
    ]

    # ==============================
    # LOAD
    # ==============================

    df = load_csvs(csv_files)

    # ==============================
    # FILTRO STATUS
    # ==============================

    if "status" in df.columns:

        df_ok = df[df["status"] == "ok"].copy()

    else:

        print("⚠️ Coluna 'status' não encontrada. Usando todas as linhas.")
        df_ok = df.copy()

    if df_ok.empty:
        print("Nenhuma linha válida encontrada.")
        return

    # ==============================
    # NORMALIZA NOMES DE COLUNAS
    # ==============================

    rename_map = {
        "modelo": "model",
        "erro": "abs_error",
        "tempo_por_amostra": "time_per_sample",
    }

    df_ok = df_ok.rename(columns=rename_map)

    required_cols = [
        "model",
        "abs_error",
        "time_per_sample",
    ]

    for col in required_cols:
        if col not in df_ok.columns:
            raise ValueError(f"Coluna obrigatória ausente: {col}")

    # ==============================
    # SUMMARY
    # ==============================

    summary = (
        df_ok.groupby("model", as_index=False)
        .agg(
            mean_abs_error=("abs_error", "mean"),
            median_abs_error=("abs_error", "median"),
            mean_time=("time_per_sample", "mean"),
            n_samples=("abs_error", "size"),
        )
        .sort_values("mean_abs_error")
    )

    print("\n===== RESULTADOS =====")
    print(summary)

    # ==============================
    # SAVE
    # ==============================

    output_path = os.path.join(
        base_dir,
        "combined_summary.csv"
    )

    summary.to_csv(output_path, index=False)

    print(f"\n✅ Summary salvo em: {output_path}")
    print(f"Total de linhas analisadas: {len(df_ok)}")


if __name__ == "__main__":
    main()

📂 Lendo: /var/new_homes/julio/mestrado/mestrado-dyssyn/experiments/exp_029/moss_calibrated_results.csv
📂 Lendo: /var/new_homes/julio/mestrado/mestrado-dyssyn/experiments/exp_029/quadapt.csv
⚠️ Coluna 'status' não encontrada. Usando todas as linhas.

===== RESULTADOS =====
                     model  mean_abs_error  median_abs_error  mean_time  \
5               DyS_Topsoe        0.049128          0.023971   0.001486   
3  C_Divergence_CALIBRATED        0.056783          0.031677   0.000809   
4      D_QDeriv_CALIBRATED        0.058450          0.032800   0.000846   
2      Baseline_CALIBRATED        0.059204          0.032437   0.000835   
1        B_Tail_CALIBRATED        0.059764          0.034153   0.000831   
0       A_Shape_CALIBRATED        0.060943          0.033746   0.000870   
6              QuaDapt_DyS        0.169672          0.061633   0.002164   

   n_samples  
5      17100  
3      17100  
4      17100  
2      17100  
1      17100  
0      17100  
6      17100  

✅ Sum

In [8]:
import pandas as pd
import plotly.express as px

# =========================================================
# LOAD
# =========================================================
results_path = Path("")

csv_files = list(results_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum CSV encontrado na pasta ../results")

dfs = []
for csv in csv_files:
    df_tmp = pd.read_csv(csv)
    df_tmp["source_file"] = csv.name  # opcional: rastrear origem
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# 🔧 garantir que erro é numérico
df["erro"] = pd.to_numeric(df["erro"], errors="coerce")

# =========================================================
# AGREGAÇÃO (modelo × dataset)
# 👉 troque mean por median se quiser
# =========================================================
df_agg = (
    df
    .groupby(["modelo", "dataset"], as_index=False)
    .agg(erro_mean=("erro", "mean"))
)

# =========================================================
# ORDEM DOS MODELOS (melhor → pior)
# =========================================================
order = (
    df_agg
    .groupby("modelo")["erro_mean"]
    .median()
    .sort_values()
    .index
    .tolist()
)


# =========================================================
# LINEPLOT
# =========================================================
fig = px.line(
    df_agg,
    x="dataset",
    y="erro_mean",
    color="modelo",
    category_orders={"modelo": order},
    markers=True
)

fig.update_layout(
    title="Erro médio por dataset (lineplot)",
    xaxis_title="Dataset",
    yaxis_title="Erro médio |prev_pred − prev_real|",
    template="simple_white",
    width=1100,
    height=500,
)

fig.update_traces(
    marker=dict(size=6),
    line=dict(width=2)
)

fig

In [10]:
# ============================
# TABELA DE MEDIANAS
# ============================

tabela_mediana = (
    df.groupby(
        ["dataset", "modelo"]
    )["erro"]
    .median()
    .reset_index()
)

# pivotar para ficar bonito
tabela_mediana = tabela_mediana.pivot(
    index="dataset",
    columns="modelo",
    values="erro"
)

print("\n===== MEDIANA POR DATASET =====\n")

print(
    tabela_mediana.round(4)
)

# salvar CSV
tabela_mediana.to_csv(
    "medianas_por_dataset.csv"
)

print(
    "\n✅ Tabela salva em medianas_por_dataset.csv"
)


===== MEDIANA POR DATASET =====

modelo         A_Shape_CALIBRATED  B_Tail_CALIBRATED  Baseline_CALIBRATED  \
dataset                                                                     
balance.1                  0.0242             0.0250               0.0278   
balance.3                  0.0222             0.0233               0.0217   
breast-cancer              0.0121             0.0145               0.0213   
cmc.1                      0.0629             0.0669               0.0661   
cmc.2                      0.0930             0.0900               0.0863   
cmc.3                      0.1078             0.1115               0.1092   
ctg.1                      0.0193             0.0197               0.0212   
ctg.2                      0.0428             0.0377               0.0321   
ctg.3                      0.0108             0.0117               0.0126   
german                     0.0509             0.0555               0.0531   
haberman                   0.1355         